# Homework 10 — Spark Structured Streaming
Michelle Silveira

This notebook covers:
- **Part 1**: Creating a streaming data source with `rate`, applying transformations, writing to memory.
- **Part 2**: Fitting a pipeline on a static CSV, then using it to transform streaming CSV files dropped into a folder.

Works on both **JupyterHub** (PySpark pre-installed) and **Google Colab** (installs PySpark automatically).

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('HW10_Streaming') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')  # suppress INFO noise
print('Spark version:', spark.version)
print('Session ready\!')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 16:31:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 16:31:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/21 16:31:04 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/21 16:31:04 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark version: 4.0.1
Session ready\!


---
# Part 1 — Creating Streaming Data Using `rate`

The `rate` source generates rows automatically at a steady pace.  
Each row has two columns: `timestamp` and `value` (an incrementing integer).

We will add two derived columns:
- `sqrt_value` — square root of `value`
- `mod4_value` — `value mod 4`

In [2]:
from pyspark.sql.functions import col, sqrt

# Create the streaming source
rate_stream = spark.readStream.format('rate').load()

# Apply transformations
rate_transformed = rate_stream.select(
    col('timestamp'),
    col('value'),
    sqrt(col('value')).alias('sqrt_value'),
    (col('value') % 4).alias('mod4_value')
)

print('Stream schema:')
rate_transformed.printSchema()

Stream schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- sqrt_value: double (nullable = true)
 |-- mod4_value: long (nullable = true)



We write the stream to an **in-memory table** called `rate_data`.  
The query starts running in the background immediately.

In [3]:
query = rate_transformed.writeStream \
    .format('memory') \
    .queryName('rate_data') \
    .start()

print('Query started\! Name:', query.name)
print('Is active:', query.isActive)

26/04/21 16:32:52 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-193e36ac-5b0f-4865-b971-e412fc371427. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:32:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query started\! Name: rate_data
Is active: True


We wait 30 seconds while the stream collects rows into the in-memory table.  
A progress counter prints every 5 seconds so you can confirm it's working.

In [4]:
import time

print('Waiting 30 seconds for data to accumulate...')
for i in range(6):
    time.sleep(5)
    count = spark.sql('SELECT count(*) as n FROM rate_data').collect()[0]['n']
    print(f'  {5*(i+1)}s elapsed — rows so far: {count}')
print('Done waiting\!')

Waiting 30 seconds for data to accumulate...
  5s elapsed — rows so far: 46
  10s elapsed — rows so far: 52
  15s elapsed — rows so far: 57
  20s elapsed — rows so far: 62
  25s elapsed — rows so far: 67
  30s elapsed — rows so far: 72
Done waiting\!


Stop the stream and query the full in-memory table with `spark.sql()`.

In [20]:
query.stop()
print('Query stopped.')

# Show all rows collected during the 30 seconds
spark.sql('SELECT * FROM rate_data').show(200)

Query stopped.
+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-21 16:32:...|    0|               0.0|         0|
|2026-04-21 16:32:...|    1|               1.0|         1|
|2026-04-21 16:32:...|    2|1.4142135623730951|         2|
|2026-04-21 16:32:...|    3|1.7320508075688772|         3|
|2026-04-21 16:32:...|    4|               2.0|         0|
|2026-04-21 16:32:...|    5|  2.23606797749979|         1|
|2026-04-21 16:32:...|    6| 2.449489742783178|         2|
|2026-04-21 16:32:...|    7|2.6457513110645907|         3|
|2026-04-21 16:33:...|    8|2.8284271247461903|         0|
|2026-04-21 16:33:...|    9|               3.0|         1|
|2026-04-21 16:33:...|   10|3.1622776601683795|         2|
|2026-04-21 16:33:...|   11|   3.3166247903554|         3|
|2026-04-21 16:33:...|   12|3.4641016151377544|         0|
|2026-04-21 16:33:...|   13| 3.6055512754

---
# Part 2 — Using CSV Data with a Pipeline

The goal here is to fit a transformation pipeline on a static training CSV, then hook that
fitted pipeline into a streaming reader that watches a folder. Each time a new CSV file
lands in the folder, the pipeline transforms it automatically and prints the result to the console.

## Step 1 - Load the Training File

Reading the instructor-provided `bikeDetails_for_fit.csv` directly into a Spark DataFrame.

In [23]:
bike_fit = spark.read.csv(
    'bikeDetails_for_fit.csv',
    header=True,
    inferSchema=True
)

print(f'Rows: {bike_fit.count()}')
bike_fit.show(5)
print('Schema:', bike_fit.schema)

Rows: 758
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|             NULL|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|             NULL|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|           148114|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|            89643|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|             NULL|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
only showing top 5 rows
Schema: StructType([StructField('name', StringType(), True), StructField('selling_price', IntegerType(), True), StructField('year', Intege

## Step 2 - SQLTransformer

Using `SQLTransformer` with the exact statement from the homework to log-transform the price and km,
rename the response to `label`, and create the `one_owner` indicator from the `owner` string column.

In [24]:
from pyspark.ml.feature import SQLTransformer

sql_trans = SQLTransformer(
    statement="""
        SELECT log(selling_price) as label,
               year,
               log(km_driven) as log_km_driven,
               CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
        FROM __THIS__
    """
)

# Quick sanity check on the training data
sql_trans.transform(bike_fit).show(5)

+------------------+----+------------------+---------+
|             label|year|     log_km_driven|one_owner|
+------------------+----+------------------+---------+
|12.072541252905651|2019| 5.857933154483459|        1|
|10.714417768752456|2017| 8.639410824140487|        1|
|11.918390573078392|2018| 9.392661928770137|        1|
|11.082142548877775|2015|10.043249494911286|        1|
| 9.903487552536127|2011|  9.95227771670556|        0|
+------------------+----+------------------+---------+
only showing top 5 rows


## Step 3 - VectorAssembler

MLlib models expect all predictors in a single `features` column.
Assembling `year`, `log_km_driven`, and `one_owner` into that vector.

In [25]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=['year', 'log_km_driven', 'one_owner'],
    outputCol='features',
    handleInvalid='keep'
)

assembler.transform(
    sql_trans.transform(bike_fit)
).show(5)

+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
|12.072541252905651|2019| 5.857933154483459|        1|[2019.0,5.8579331...|
|10.714417768752456|2017| 8.639410824140487|        1|[2017.0,8.6394108...|
|11.918390573078392|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|11.082142548877775|2015|10.043249494911286|        1|[2015.0,10.043249...|
| 9.903487552536127|2011|  9.95227771670556|        0|[2011.0,9.9522777...|
+------------------+----+------------------+---------+--------------------+
only showing top 5 rows


## Step 4 - Build and Fit the Pipeline

Chaining the two stages into a `Pipeline` and fitting it on the training data.
The key advantage here is that this fitted pipeline object can be applied directly
to the streaming DataFrame later - no need to redo the transformations manually.

In [26]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[sql_trans, assembler])
fitted_pipeline = pipeline.fit(bike_fit)

print('Pipeline fitted\!')
fitted_pipeline.transform(bike_fit).select('label', 'features').show(5)

Pipeline fitted\!
+------------------+--------------------+
|             label|            features|
+------------------+--------------------+
|12.072541252905651|[2019.0,5.8579331...|
|10.714417768752456|[2017.0,8.6394108...|
|11.918390573078392|[2018.0,9.3926619...|
|11.082142548877775|[2015.0,10.043249...|
| 9.903487552536127|[2011.0,9.9522777...|
+------------------+--------------------+
only showing top 5 rows


## Step 5 - Prepare the Folders

Setting up two folders:
- `bike_staging/` holds the five instructor add files before they are streamed
- `bike_stream/` is the folder the stream watches - it must be **empty** when the stream starts

The add files must already be uploaded to JupyterHub before running this cell.



In [37]:
import os, shutil, codecs

STAGING_DIR = 'bike_staging'
STREAM_DIR  = 'bike_stream'

os.makedirs(STAGING_DIR, exist_ok=True)
if os.path.exists(STREAM_DIR):
    shutil.rmtree(STREAM_DIR)
os.makedirs(STREAM_DIR)

# Strip BOM while copying to staging
for i in range(1, 6):
    fname = f'bikeDetails_add{i}.csv'
    with codecs.open(fname, 'r', encoding='utf-8-sig') as f_in:
        content = f_in.read()
    with open(os.path.join(STAGING_DIR, fname), 'w', encoding='utf-8') as f_out:
        f_out.write(content)
    print(f'Staged {fname} (BOM stripped)')

print('Stream folder empty:', os.listdir(STREAM_DIR) == [])

Staged bikeDetails_add1.csv (BOM stripped)
Staged bikeDetails_add2.csv (BOM stripped)
Staged bikeDetails_add3.csv (BOM stripped)
Staged bikeDetails_add4.csv (BOM stripped)
Staged bikeDetails_add5.csv (BOM stripped)
Stream folder empty: True


## Step 6 - Set Up and Start the readStream

Using the schema from `bike_fit` so Spark knows the column types ahead of time.
The `UTF-8-BOM` encoding option handles the BOM character present in the instructor's add files.
The fitted pipeline transforms each micro-batch as it arrives, and results print to the console.

In [38]:
bike_schema = bike_fit.schema

stream_input = spark.readStream \
    .schema(bike_schema) \
    .option('header', 'true') \
    .csv(STREAM_DIR)

stream_output = fitted_pipeline.transform(stream_input)

stream_query = stream_output.writeStream \
    .format('console') \
    .outputMode('append') \
    .option('truncate', 'false') \
    .start()

print('Stream query started!')
print('Watching folder:', STREAM_DIR)

Stream query started!
Watching folder: bike_stream


26/04/21 17:16:30 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-81368115-17c9-433c-98b0-602314376139. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 17:16:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## Step 7 - Add Files One at a Time

Running each cell below individually. After each copy the console output from the cell above
should show a new batch with the transformed rows.

In [39]:
import time, shutil, os

fname = 'bikeDetails_add1.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done. Check console output.')

Copied bikeDetails_add1.csv -> bike_stream/
-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|8.987196820661973 |2003|10.887436932884098|1        |[2003.0,10.887436932884098,1.0]|
|11.156250521031495|2018|9.615805480084347 |1        |[2018.0,9.615805480084347,1.0] |
|10.819778284410283|2016|8.987196820661973 |1        |[2016.0,8.987196820661973,1.0] |
|10.46310334047155 |2015|10.582738627903963|1        |[2015.0,10.582738627903963,1.0]|
|9.903487552536127 |2006|11.225243392518447|1        |[2006.0,11.225243392518447,1.0]|
|10.819778284410283|2012|10.239959789157341|1        |[2012.0,10.239959789157341,1.0]|
|10.51867319162636 |2008|11.03488966402723 |1        |[2008.0,11.03488966402

Add File 2

In [40]:
fname = 'bikeDetails_add2.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done.')

Copied bikeDetails_add2.csv -> bike_stream/
-------------------------------------------
Batch: 1
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|10.46310334047155 |2011|10.239959789157341|1        |[2011.0,10.239959789157341,1.0]|
|10.714417768752456|2015|10.596634733096073|1        |[2015.0,10.596634733096073,1.0]|
|11.49272275765271 |2019|9.047821442478408 |1        |[2019.0,9.047821442478408,1.0] |
|11.225243392518447|2017|10.308952660644293|1        |[2017.0,10.308952660644293,1.0]|
|10.915088464214607|2017|9.903487552536127 |0        |[2017.0,9.903487552536127,0.0] |
|10.126631103850338|2013|10.308952660644293|1        |[2013.0,10.308952660644293,1.0]|
|11.156250521031495|2014|9.438750032962892 |1        |[2014.0,9.438750032962

Add File 3

In [41]:
fname = 'bikeDetails_add3.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done.')

Copied bikeDetails_add3.csv -> bike_stream/
-------------------------------------------
Batch: 2
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|10.51867319162636 |2016|10.714417768752456|1        |[2016.0,10.714417768752456,1.0]|
|10.645424897265505|2013|10.545341438708522|1        |[2013.0,10.545341438708522,1.0]|
|11.686878772093667|2016|9.893437216682626 |1        |[2016.0,9.893437216682626,1.0] |
|12.611537753638338|2011|9.392661928770137 |1        |[2011.0,9.392661928770137,1.0] |
|11.77528972943772 |1993|9.210340371976184 |0        |[1993.0,9.210340371976184,0.0] |
|9.998797732340453 |2008|11.350406535472453|0        |[2008.0,11.350406535472453,0.0]|
|9.615805480084347 |2007|11.156250521031495|1        |[2007.0,11.15625052103

Add File 4

In [42]:
fname = 'bikeDetails_add4.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done.')

Copied bikeDetails_add4.csv -> bike_stream/
-------------------------------------------
Batch: 3
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|10.596634733096073|2015|9.210340371976184 |1        |[2015.0,9.210340371976184,1.0] |
|10.308952660644293|2011|11.396391648714276|1        |[2011.0,11.396391648714276,1.0]|
|9.95227771670556  |2009|10.819778284410283|1        |[2009.0,10.819778284410283,1.0]|
|10.46310334047155 |2011|9.648595302907339 |1        |[2011.0,9.648595302907339,1.0] |
|10.043249494911286|2011|11.097410021008562|1        |[2011.0,11.097410021008562,1.0]|
|10.596634733096073|2017|11.289781913656018|0        |[2017.0,11.289781913656018,0.0]|
|10.596634733096073|2011|9.0595174822416   |1        |[2011.0,9.059517482241

Add File 5

In [43]:
fname = 'bikeDetails_add5.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done. All 5 files processed\!')

Copied bikeDetails_add5.csv -> bike_stream/
-------------------------------------------
Batch: 4
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|10.915088464214607|2017|10.968198289528557|1        |[2017.0,10.968198289528557,1.0]|
|10.858998997563564|2012|10.714417768752456|1        |[2012.0,10.714417768752456,1.0]|
|10.839580911706463|2018|10.085809109330082|1        |[2018.0,10.085809109330082,1.0]|
|10.819778284410283|2013|8.699514748210191 |1        |[2013.0,8.699514748210191,1.0] |
|10.819778284410283|2018|10.341742483467284|1        |[2018.0,10.341742483467284,1.0]|
|10.819778284410283|2014|9.472704636443673 |1        |[2014.0,9.472704636443673,1.0] |
|10.819778284410283|2015|10.714417768752456|1        |[2015.0,10.71441776875

Stop the Stream Query

Once all files are processed and console output has appeared, stop the query.

## Step 8 - Stop the Stream

All five files have been added and each batch appeared in the console output. Stopping the query now.

In [36]:
stream_query.stop()
print('Stream query stopped.')
print('Active streams remaining:', spark.streams.active)

Stream query stopped.
Active streams remaining: []
